# LAWM Entry Point Notebook

Notebook wrapper for `app/main.py` so you can launch training/inference without CLI.

In [1]:
from pathlib import Path
import os, sys

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

Project root: /mnt/HyperStorage_ntfs/Coding/Autonomous/Simulation/model/LAWM


In [ ]:
# --- Configure launch args as a Python dictionary (entangled like YAML anchors) ---
COMMON_SETTINGS = {
    "patch_size": 16,
    "tubelet_size": 2,
    "crop_size": 384,
    "nclips": 1,
    "fpcs": 16,
}

LOADER_SETTINGS = {
    "num_workers": 4,
    "persistent_workers": True,
    "pin_mem": True,
}

PARAMS = {
    "common": {**COMMON_SETTINGS},
    "app": "probe",
    "logging": {
        "progress_type": "table",
        "save_csv": True,
        "save_batch_csv": True,
        "save_epoch_csv": True,
    },
    "loader_setup": {**LOADER_SETTINGS},
    "train": {
        **COMMON_SETTINGS,
        **LOADER_SETTINGS,
        "batch_size": 8,
        "allow_clip_overlap": True,
        "random_jiggle": True,
        "train_fraction": 0.85,
        "val_fraction": 0.15,
        "datasets": [
            {"name": "carla", "path": "./csv_metadata/probe/Carla_recording_20251025_142727_best_spatial.csv", "fps": 4},
            {"name": "carla", "path": "./csv_metadata/probe/Carla_recording_20260204_090029_spatial.csv", "fps": 4},
            {"name": "carla", "path": "./csv_metadata/probe/Carla_recording_20260318_083409_best_spatial.csv", "fps": 4},
            {"name": "carla", "path": "./csv_metadata/probe/Carla_recording_20260317_214033_spatial.csv", "fps": 4},
            {"name": "carla", "path": "./csv_metadata/probe/Carla_recording_20260317_233603_spatial.csv", "fps": 4},
            {"name": "carla", "path": "./csv_metadata/probe/Carla_recording_20260321_152022_best_spatial.csv", "fps": 4},
            {"name": "carla", "path": "./csv_metadata/probe/Carla_recording_20260308_212005_spatial.csv", "fps": 4},
            {"name": "carla", "path": "./csv_metadata/probe/Carla_recording_20260323_204100_best_spatial.csv", "fps": 4},
            {"name": "carla", "path": "./csv_metadata/probe/Carla_recording_20260323_210357_best_spatial.csv", "fps": 4},
        ],
    },
    "data_aug": {
        "auto_augment": False,
        "horizontal_flip": False,
        "motion_shift": False,
        "random_resize_aspect_ratio": [1.0, 1.0],
        "random_resize_scale": [1.0, 1.0],
        "reprob": 0.0,
    },
    "meta": {
        "dtype": "bfloat16",
        "save_every_freq": 5,
        "save_root_dir": "./Experiment",
        "seed": 239,
        "sync_gc": True,
        "resume_prefer_best": True,
    },
    "model": {
        "compile": False,
        "common": {
            "use_sdpa": True,
        },
        "enc": {
            **COMMON_SETTINGS,
            "name": "vjepa2_1_vit_base_384",
            "load_from": "facebookresearch/vjepa2",
            "source": "github",
            "use_activation_checkpointing": True,
        },
        "probe": {
            "name": "EfficientProbe",
            "output_dim": 2,
            "hidden_dim": None,
            "max_frames": COMMON_SETTINGS["fpcs"],
            "tubelet_size": COMMON_SETTINGS["tubelet_size"],
            "num_heads": 8,
            "num_queries": 16,
            "depth": 1,
            "mlp_ratio": 4.0,
            "init_std": 0.04,
            "qkv_bias": True,
            "dropout": 0.15,
            "use_activation_checkpointing": True,
            "init_scales": None,
            "init_shifts": None,
        },
    },
    "optimization": {
        "anneal": 75,
        "epochs": 100,
        "warmup": 10,
        "final_lr": 0.0,
        "final_weight_decay": 0.04,
        "ipe": 150,
        "lr": 0.000225,
        "start_lr": 0.000045,
        "weight_decay": 0.04,
        "gradient_optimizer": {
            "type": "normal",
            "params": {
                "reduction": "mean",
            },
        },
    },
    "loss": {
        "enable_velocity": True,
        "enable_steer": False,
        "enable_lateral_error": True,
        "velocity_loss": "log_cosh",
        "steer_loss": "smooth_l1",
        "lateral_error_loss": "log_cosh",
        "velocity_idx": 0,
        "steer_idx": 1,
        "lateral_error_idx": 1,
        "velocity_log_var": 0.0,
        "steer_log_var": 0.0,
        "lateral_error_log_var": 0.0,
        "reduction": "mean",
    },
}

DEVICES = ["cuda:0"]  # e.g. ["cuda:0", "cuda:1"]

print("Config loaded as dict (PARAMS)")
print("Progress mode:", PARAMS["logging"]["progress_type"] )
print("Devices:", DEVICES)
print("Entangled common settings:", COMMON_SETTINGS)

Config loaded as dict (PARAMS)
Progress mode: table
Devices: ['cuda:0']
Entangled common settings: {'patch_size': 16, 'tubelet_size': 2, 'crop_size': 384, 'nclips': 1, 'fpcs': 16}


In [3]:
import os
import importlib
import multiprocessing as mp
import torch

from utils.distributed import init_distributed
from utils.logger import Logger

def _process_with_params(rank, params, world_size, devices):
    os.environ["CUDA_VISIBLE_DEVICES"] = str(devices[rank].split(":")[-1])
    logger = Logger()
    logger.INFO(f"Rank {rank} loaded parameters from dict")

    world_size, rank = init_distributed(rank_and_world_size=(rank, world_size))
    if torch.distributed.is_available() and torch.distributed.is_initialized() and world_size > 1:
        logger.CUSTOM("SUCCESS", f"DDP enabled (world_size={world_size}, rank={rank})")
    else:
        logger.INFO("DDP disabled (single-GPU/single-process mode)")

    if rank == 0:
        Logger.set_levels("INFO", "ERROR", "WARNING", "DEBUG", "CUSTOM")
    else:
        Logger.set_levels("ERROR", "DEBUG", "CUSTOM")

    try:
        importlib.import_module(f"app.{params['app']}.train").main(params, "main.ipynb")
    except KeyboardInterrupt:
        logger.ERROR(f"Keyboard Interrupt detected on rank={rank}")
    except Exception as e:
        logger.ERROR(f"Error on rank={rank}", full_traceback=e)
    finally:
        if torch.distributed.is_available() and torch.distributed.is_initialized():
            if torch.distributed.get_world_size() > 1:
                torch.distributed.barrier()
            torch.distributed.destroy_process_group()
        logger.DEBUG(f"Destroyed process group on rank={rank}")

def launch_entrypoint(params: dict, devices: list[str]):
    world_size = len(devices)
    if world_size < 1:
        raise ValueError("devices must contain at least one device")

    if world_size == 1:
        _process_with_params(0, params, world_size, devices)
        return

    ctx = mp.get_context("spawn")
    procs = []
    for rank in range(world_size):
        p = ctx.Process(target=_process_with_params, args=(rank, params, world_size, devices))
        p.start()
        procs.append(p)

    for p in procs:
        p.join()

    failed = [(idx, p.exitcode) for idx, p in enumerate(procs) if p.exitcode != 0]
    if failed:
        raise RuntimeError(f"One or more ranks failed: {failed}")

In [4]:
# --- Run entrypoint with dict config ---
launch_entrypoint(PARAMS, DEVICES)

[2026/03/23-22:32:24] [INFO]    [ipykernel.zmqshell.ZMQInteractiveShell]: Rank 0 loaded parameters from dict

[2026/03/23-22:32:24] [INFO]    [ipykernel.zmqshell.ZMQInteractiveShell]: DDP disabled (single-GPU/single-process 
mode)

[2026/03/23-22:32:31] [INFO]    [utils.distributed]: SLURM vars not set (distributed training not available)

[2026/03/23-22:32:31] [INFO]    [app.probe.train]: DDP disabled (single-GPU/single-process mode)

[2026/03/23-22:32:32] [INFO]    [app.probe.compile.models]: Loading the model from github

...

[2026/03/23-22:32:34] [INFO]    [app.probe.compile.optim]: Gradient optimizer initialized: NORMAL

[2026/03/23-22:32:34] [INFO]    [app.probe.train]: Probe save root directory: ./Experiment/probe

╭──────────╮
│  Epoch   │
├──────────┤
╭──────────┬──────────┬──────────┬───────────┬────────────────────┬──────────────────┬──────────────────┬────────────────┬──────────────────────┬────────────────────┬──────────────────┬──────────────────────╮
│  Epoch   │    LR    │    WD    │ GPU Timer │ Train | Total Loss │ Val | Total Loss │ Train | Velocity │ Val | Velocity │ Train | LateralError │ Val | LateralError │ VelocityWeighted │ LateralErrorWeighted │
├──────────┼──────────┼──────────┼───────────┼────────────────────┼──────────────────┼──────────────────┼────────────────┼──────────────────────┼────────────────────┼──────────────────┼──────────────────────┤
│  1/100   │ 4.51e-05 │  0.0400  │ 4916.2305 │      23.0708       │                  │     22.8961      │                │        0.1746        │                    │     22.8961      │        0.1746        │
│  1/100   │ 4.52e-05 │  0.0400  │ 3074.8710 │      29.9531       │                  │     29.7923      │                │   

[2026/03/23-22:33:04] [ERROR]   [ipykernel.zmqshell.ZMQInteractiveShell]: Keyboard Interrupt detected on rank=0

[2026/03/23-22:33:04] [DEBUG]   [ipykernel.zmqshell.ZMQInteractiveShell]: Destroyed process group on rank=0